# Mini-projet de synthèse — cumulatif S1 → S3

**Contexte.** Tu récupères l'historique de ventes d'un petit réseau de pharmacies :
`data/ventes_pharmacies.csv` (date, pharmacie, médicament, classe thérapeutique, prix unitaire,
remboursable, quantité). On te demande une petite analyse de bout en bout.

**Objectif (ce que tu dois en retenir).** C'est ta **répétition générale** avant les vrais projets :
enchaîner un mini-pipeline d'analyse en **combinant tout depuis la S1** —
charger/inspecter proprement (S1), manipuler avec pandas : filtres, `value_counts`, colonne calculée,
`.str` (S2), **agréger avec `groupby` en restant lucide sur les NaN** (S3), **encapsuler tes calculs
dans une classe** qui *renvoie* ses résultats (S3), et répondre à des questions **SQL** mêlant jointure,
sous-requête et window function (S1→S3).

**Méthode.** Questions formulées comme des besoins métier — **à toi de choisir tes outils**. Commente,
et termine par une **synthèse écrite**.

## Partie A — Charger et inspecter (S1 + S2, vigilance NaN de la S3)
Charge le fichier. Combien de ventes ? Quelles colonnes/types ? **Une colonne a des valeurs manquantes : laquelle, combien ?**

In [115]:
import pandas as pd

df = pd.read_csv("data/ventes_pharmacies.csv")
df["date"] = pd.to_datetime(df["date"])
distinct_sells = df["vente_id"].nunique()

## --- Code sans Markdown mais avec des df plus lisibles 
print("- Dataframe :")
display(df)  # S'affichera sous forme de joli tableau HTML

print(f"\n- Il y a eu en tout {distinct_sells} ventes\n")

print("- Type des colonnes :")
display(df.dtypes.to_frame(name="Type"))  # .to_frame() force un joli tableau pour les fonctions / calculs qui affichent un résultat 
                                          # "sans colonnes" ici la colonne des dtypes ou celle plus bas qui calculent les NaN dans les colonnes 
                                          # Renvoient une colonne vide de nom alors qu'avec to_frame ça s'ajoute au df

print("\n- Description du df :")
display(df.describe())

print("\n- Total valeurs NaN :")
display(df.isna().sum().to_frame(name="Nb NaN"))

# -- Code avec Markdown --

# import pandas as pd
# from IPython.display import display, Markdown

# # Ton code de traitement reste le même...

# # Affichage avec des vrais titres en gras
# display(Markdown("### 📊 Extrait du Dataframe :"))
# display(df.head()) # .head() pour éviter d'afficher 120 lignes d'un coup

# display(Markdown(f"**💡 Nombre de ventes distinctes :** {distinct_sells}"))

# display(Markdown("### 🔍 Analyse des colonnes et des NaN :"))
# display(df.isna().sum().to_frame(name="Total NaN"))

- Dataframe :


,vente_id,date,pharmacie,medicament,classe_therapeutique,prix_unitaire,remboursable,quantite
0,V0047,2026-01-01,Grande Pharmacie,Amlodipine 5mg,Antihypertenseur,4.30,Oui,27
1,V0069,2026-01-01,Grande Pharmacie,Levothyrox 75µg,Hormone thyroïdienne,3.37,Oui,35
2,V0048,2026-01-02,Pharmacie de la Gare,Vitamine D3,Supplément,6.25,Non,7
3,V0117,2026-01-02,NaN,Amoxicilline 1g,Antibiotique,5.09,Oui,7
4,V0071,2026-01-03,Grande Pharmacie,Ventoline 100µg,Bronchodilatateur,4.52,Oui,10
...,...,...,...,...,...,...,...,...
115,V0090,2026-06-25,Pharmacie des Halles,Gaviscon menthe,Antiacide,5.33,Non,32
116,V0045,2026-06-26,Pharmacie du Centre,Gaviscon menthe,Antiacide,5.26,Non,12
117,V0064,2026-06-28,Pharmacie Saint-Marc,Augmentin 1g,Antibiotique,7.97,Oui,26
118,V0094,2026-06-28,Grande Pharmacie,Doliprane 1000mg,Antalgique,2.17,Oui,18



- Il y a eu en tout 120 ventes

- Type des colonnes :


vente_id                           str
date                    datetime64[us]
pharmacie                          str
medicament                         str
classe_therapeutique               str
prix_unitaire                  float64
remboursable                       str
quantite                         int64
dtype: object


- Description du df :


,date,prix_unitaire,quantite
count,120,120.000000,120.000000
mean,2026-04-03 02:24:00,6.911167,21.791667
min,2026-01-01 00:00:00,2.030000,1.000000
25%,2026-02-18 00:00:00,4.142500,11.000000
50%,2026-04-08 00:00:00,4.565000,22.500000
75%,2026-05-17 06:00:00,5.585000,33.000000
max,2026-06-28 00:00:00,39.510000,40.000000
std,NaN,8.680770,11.563414



- Total valeurs NaN :


vente_id                0
date                    0
pharmacie               8
medicament              0
classe_therapeutique    0
prix_unitaire           0
remboursable            0
quantite                0
dtype: int64

In [98]:
df["pharmacie"] = df["pharmacie"].fillna("Pharmacie inconnue")
df

,vente_id,date,pharmacie,medicament,classe_therapeutique,prix_unitaire,remboursable,quantite
0,V0047,2026-01-01,Grande Pharmacie,Amlodipine 5mg,Antihypertenseur,4.30,Oui,27
1,V0069,2026-01-01,Grande Pharmacie,Levothyrox 75µg,Hormone thyroïdienne,3.37,Oui,35
2,V0048,2026-01-02,Pharmacie de la Gare,Vitamine D3,Supplément,6.25,Non,7
3,V0117,2026-01-02,Pharmacie inconnue,Amoxicilline 1g,Antibiotique,5.09,Oui,7
4,V0071,2026-01-03,Grande Pharmacie,Ventoline 100µg,Bronchodilatateur,4.52,Oui,10
...,...,...,...,...,...,...,...,...
115,V0090,2026-06-25,Pharmacie des Halles,Gaviscon menthe,Antiacide,5.33,Non,32
116,V0045,2026-06-26,Pharmacie du Centre,Gaviscon menthe,Antiacide,5.26,Non,12
117,V0064,2026-06-28,Pharmacie Saint-Marc,Augmentin 1g,Antibiotique,7.97,Oui,26
118,V0094,2026-06-28,Grande Pharmacie,Doliprane 1000mg,Antalgique,2.17,Oui,18


## Partie B — L'indicateur métier (S2 + S3)
Crée une colonne `montant` (prix × quantité). Donne le **chiffre d'affaires (CA) total** — et fais bien attention à ne pas te faire piéger par les valeurs manquantes.

In [99]:
# ton code
df["montant"] = df["prix_unitaire"] * df["quantite"]

chiffre_affaire = df["montant"].sum()


print(chiffre_affaire, "\n\n")


19124.469999999998 




## Partie C — Profiler (S2)
Combien de ventes **par pharmacie** ? **par classe thérapeutique** ? Quelle est la part des ventes **remboursables** ?

In [100]:
# ton code
ventes_pharma = df.groupby("pharmacie")["vente_id"].count().sort_values()
ventes_classe = df.groupby("classe_therapeutique")["vente_id"].count().sort_values()
part_remboursable = (df["remboursable"].value_counts(normalize=True))*100

print(ventes_pharma, "\n\n")
print(ventes_classe, "\n\n")
print("Part de médicament remboursable :\n", part_remboursable, "\n\n")

pharmacie
Pharmacie inconnue       8
Pharmacie des Halles    17
Pharmacie de la Gare    18
Pharmacie du Centre     23
Grande Pharmacie        26
Pharmacie Saint-Marc    28
Name: vente_id, dtype: int64 


classe_therapeutique
Antalgique               5
Antispasmodique          7
Antiacide                8
Hormone thyroïdienne     8
Anti-inflammatoire       9
ORL                     10
Antihypertenseur        11
Supplément              14
Antidiabétique          15
Antibiotique            16
Bronchodilatateur       17
Name: vente_id, dtype: int64 


Part de médicament remboursable :
 remboursable
Oui    67.5
Non    32.5
Name: proportion, dtype: float64 




## Partie D — Agréger (S3)
1. Le **CA par classe thérapeutique** (classement décroissant).
2. Le **CA par pharmacie**.
3. Le **top 3 des médicaments** par CA.

In [101]:
# ton code
chiffre_affaire_pharmacie = df.groupby("pharmacie")["montant"].sum().sort_values(ascending=False)
chiffre_affaire_classe = df.groupby("classe_therapeutique")["montant"].sum().sort_values(ascending=False)
chiffre_affaire_medicament = df.groupby("medicament")["montant"].sum()
top_medicament = df.groupby("medicament")["montant"].sum().nlargest(n=3)

print(chiffre_affaire_pharmacie, "\n\n")
print(chiffre_affaire_classe, "\n\n")
print(chiffre_affaire_medicament, "\n\n")
print(top_medicament, "\n\n")

pharmacie
Pharmacie des Halles    4225.21
Grande Pharmacie        3992.02
Pharmacie de la Gare    3321.33
Pharmacie du Centre     3004.19
Pharmacie Saint-Marc    2899.48
Pharmacie inconnue      1682.24
Name: montant, dtype: float64 


classe_therapeutique
Antidiabétique          8580.84
Antibiotique            2149.71
Bronchodilatateur       1777.25
Supplément              1691.75
ORL                     1252.35
Antiacide               1111.73
Antihypertenseur        1046.15
Hormone thyroïdienne     672.25
Antispasmodique          358.62
Anti-inflammatoire       258.39
Antalgique               225.43
Name: montant, dtype: float64 


medicament
Amlodipine 5mg       1046.15
Amoxicilline 1g      1091.50
Augmentin 1g         1058.21
Doliprane 1000mg      225.43
Gaviscon menthe      1111.73
Ibuprofène 400mg      258.39
Lantus 100UI         7820.70
Levothyrox 75µg       672.25
Magnésium B6          837.08
Metformine 1000mg     760.14
Spasfon               358.62
Strepsils            1252.35


## Partie E — Filtrer et fouiller le texte (S2)
Isole les ventes de médicaments dosés « **1g** » (indice : accesseur `.str`). Puis, une question de ton choix combinant un filtre et un tri.

In [102]:
# ton code
df_1g = df[df["medicament"].str.contains("1g")]
df_1g

# On va isoler les médicaments avec des montants > à la moyenne et les trier par ordre ascendant 

df_moy = (
    df[df["montant"] >= df["montant"].mean()]
    .sort_values("montant")
)
df_moy

,vente_id,date,pharmacie,medicament,classe_therapeutique,prix_unitaire,remboursable,quantite,montant
17,V0072,2026-01-24,Pharmacie Saint-Marc,Amlodipine 5mg,Antihypertenseur,4.26,Oui,38,161.88
105,V0014,2026-06-10,Pharmacie de la Gare,Ventoline 100µg,Bronchodilatateur,4.54,Oui,36,163.44
85,V0043,2026-05-10,Pharmacie Saint-Marc,Ventoline 100µg,Bronchodilatateur,4.42,Oui,37,163.54
84,V0027,2026-05-10,Grande Pharmacie,Amoxicilline 1g,Antibiotique,5.10,Oui,33,168.30
115,V0090,2026-06-25,Pharmacie des Halles,Gaviscon menthe,Antiacide,5.33,Non,32,170.56
42,V0032,2026-03-15,Grande Pharmacie,Ventoline 100µg,Bronchodilatateur,4.41,Oui,39,171.99
72,V0040,2026-04-21,Pharmacie du Centre,Augmentin 1g,Antibiotique,7.85,Oui,22,172.70
74,V0100,2026-04-22,Pharmacie Saint-Marc,Magnésium B6,Supplément,7.88,Non,22,173.36
10,V0067,2026-01-12,Pharmacie des Halles,Amoxicilline 1g,Antibiotique,5.18,Oui,34,176.12
70,V0009,2026-04-17,Pharmacie des Halles,Ventoline 100µg,Bronchodilatateur,4.56,Oui,39,177.84


In [103]:
print(df[df["pharmacie"].str.startswith("Pharmacie")])

    vente_id       date             pharmacie       medicament  \
2      V0048 2026-01-02  Pharmacie de la Gare      Vitamine D3   
3      V0117 2026-01-02    Pharmacie inconnue  Amoxicilline 1g   
5      V0076 2026-01-04  Pharmacie Saint-Marc   Amlodipine 5mg   
6      V0118 2026-01-06  Pharmacie de la Gare        Strepsils   
7      V0033 2026-01-07   Pharmacie du Centre     Magnésium B6   
..       ...        ...                   ...              ...   
114    V0104 2026-06-24  Pharmacie des Halles        Strepsils   
115    V0090 2026-06-25  Pharmacie des Halles  Gaviscon menthe   
116    V0045 2026-06-26   Pharmacie du Centre  Gaviscon menthe   
117    V0064 2026-06-28  Pharmacie Saint-Marc     Augmentin 1g   
119    V0112 2026-06-28  Pharmacie Saint-Marc  Amoxicilline 1g   

    classe_therapeutique  prix_unitaire remboursable  quantite  montant  
2             Supplément           6.25          Non         7    43.75  
3           Antibiotique           5.09          Oui       

## Partie F — Encapsuler dans une classe (S3 — OOP)

Écris une classe `AnalyseVentes` qui, **à sa création**, reçoit le DataFrame des ventes (avec la colonne
`montant`). Elle doit proposer des méthodes qui **renvoient** (⚠️ `return`, pas `print`) :

1. le **CA total** ;
2. le **CA par classe thérapeutique** ;
3. le **CA d'une pharmacie donnée** (passée en argument).

Puis crée une instance et appelle tes trois méthodes. *Rappel §11 de la fiche : une méthode qui calcule
doit `return` — sinon tu ne peux pas réutiliser le résultat.*

In [104]:
# ton code : la classe AnalyseVentes
class AnalyseVentes:
    def __init__(self, df):
        self.df = df
        
    def chiffr_affaire_total (self):
        return self.df["montant"].sum()
    
    def chiffr_affaire_classe (self):
        return self.df.groupby("classe_therapeutique")["montant"].sum()
    
    def chiffr_affaire_pharmacie (self, pharmacie):
        return self.df[self.df["pharmacie"]==pharmacie].groupby("pharmacie")["montant"].sum()

In [105]:
# ton code : créer une instance et tester les 3 méthodes
a = AnalyseVentes(df)
print(a.chiffr_affaire_total(), "\n\n")
print(a.chiffr_affaire_classe(), "\n\n")
print(a.chiffr_affaire_pharmacie("Pharmacie inconnue"), "\n\n")


19124.469999999998 


classe_therapeutique
Antalgique               225.43
Anti-inflammatoire       258.39
Antiacide               1111.73
Antibiotique            2149.71
Antidiabétique          8580.84
Antihypertenseur        1046.15
Antispasmodique          358.62
Bronchodilatateur       1777.25
Hormone thyroïdienne     672.25
ORL                     1252.35
Supplément              1691.75
Name: montant, dtype: float64 


pharmacie
Pharmacie inconnue    1682.24
Name: montant, dtype: float64 




## Partie G — En SQL (S1 → S3)

La cellule ci-dessous est **fournie** : elle charge le CSV dans une base SQLite en mémoire, découpée en
deux tables (`ventes` et `medicaments`), et te donne une fonction `q("...")` qui exécute une requête et
renvoie le résultat. **Exécute-la, puis écris tes requêtes dans les cellules suivantes.**

In [106]:
# --- CELLULE FOURNIE : rien à modifier, exécute-la ---
import sqlite3, pandas as pd
_df = pd.read_csv("data/ventes_pharmacies.csv")
_con = sqlite3.connect(":memory:")
# table des ventes (le "fait")
_df[["vente_id","date","pharmacie","medicament","quantite"]].to_sql("ventes", _con, index=False, if_exists="replace")
# catalogue des médicaments (la "dimension"), une ligne par médicament
_meds = _df[["medicament","classe_therapeutique","prix_unitaire","remboursable"]].drop_duplicates("medicament")
_meds.to_sql("medicaments", _con, index=False, if_exists="replace")
def q(sql):
    return pd.read_sql(sql, _con)   # écris tes requêtes avec q("SELECT ...")
print("Tables prêtes : ventes", _df.shape[0], "lignes | medicaments", _meds.shape[0], "lignes")

Tables prêtes : ventes 120 lignes | medicaments 14 lignes


**G.1** — En joignant les deux tables, liste les ventes (id, médicament, classe) portant sur des médicaments **plus chers que le prix moyen du catalogue**. (jointure + sous-requête)

In [107]:
q("""
   SELECT v.vente_id, v.medicament, m.classe_therapeutique
   from ventes v
   join medicaments m on v.medicament = m.medicament 
   where m.prix_unitaire > (SELECT avg(prix_unitaire) as moy
                            from medicaments)
   
 """)

,vente_id,medicament,classe_therapeutique
0,V0033,Magnésium B6,Supplément
1,V0077,Magnésium B6,Supplément
2,V0086,Magnésium B6,Supplément
3,V0098,Magnésium B6,Supplément
4,V0100,Magnésium B6,Supplément
5,V0013,Augmentin 1g,Antibiotique
6,V0023,Augmentin 1g,Antibiotique
7,V0040,Augmentin 1g,Antibiotique
8,V0064,Augmentin 1g,Antibiotique
9,V0073,Augmentin 1g,Antibiotique


**G.2** — Dans **chaque classe thérapeutique**, classe les médicaments du catalogue du plus cher au moins cher et numérote-les (1 = le plus cher de sa classe). (window function)

In [110]:
q("""
   SELECT m.classe_therapeutique, m.prix_unitaire,
          RANK() OVER (PARTITION BY m.classe_therapeutique ORDER by m.prix_unitaire DESC) as Rang
   from medicaments m
   """)


,classe_therapeutique,prix_unitaire,Rang
0,Antalgique,2.23,1
1,Anti-inflammatoire,2.45,1
2,Antiacide,5.28,1
3,Antibiotique,8.07,1
4,Antibiotique,5.09,2
5,Antidiabétique,39.29,1
6,Antidiabétique,4.57,2
7,Antihypertenseur,4.30,1
8,Antispasmodique,2.06,1
9,Bronchodilatateur,4.52,1


## Partie H — Synthèse

En 4 à 6 phrases : qu'as-tu appris sur ces ventes ? (classes/pharmacies qui pèsent le plus, poids du
remboursable, médicaments clés, limites des données). C'est la partie qui compte le plus.

Depuis le début j'ai appris pas mal de choses dans ce datasets :
- Tout d'abord même si très complet on peut voir qu'il manque des données de pharmacie qu'on a remplacé par "Pharmacie inconnu".
- 2 tiers des médicament sont remboursables (67.5% remboursable contre 32.5%).
- Les médicaments suivants sont les classes qui ont le plus vendus: Supplément, Antidiabétique, Antibiotique  et Bronchodilatateur . 
- Les médicaments suivants sont ceux qui rapportent le plus : Lantus 100UI, Ventoline 100µg et Strepsils.
- Les pharmacies des Halles et de la Gare sont les plus chères moins de ventes mais respectivement top 1 et 3 en chiffre d'affaires.
- Seulement on a que des données de ventes sur les pharmacies et aucune données sur les clients, vendeurs, etc...